# Here we need to re-do the february experiment with 18hours as tp0 and use median metric instead

In [12]:
import polars as pl
from pathlib import Path
from subprocess import run

# analysis ready raw data
analysis_ready_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
    "Study/analysis_ready_data"
)

# datasets
data_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data"
)

# platemaps
pm_path = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
    "Study/platemaps/2026-03-16CellPaint_map.xlsx"
)
platemap = pl.read_excel(pm_path, engine="openpyxl")
# display(platemap)

analysis_results_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/results"
)

pickle_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/pickles")

visual_results_dir = Path(
    "/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals"
)

display(platemap)

WellName,stain set,cell line,condition
str,str,str,str
"""G12""","""Chroma""","""Bmal1""","""no treatment"""
"""G13""","""Chroma""","""Bmal1""","""no treatment"""
"""G14""","""Chroma""","""Bmal1""","""no treatment"""
"""G18""","""Chroma""","""Bmal1""","""0.1uM Dex"""
"""G19""","""Chroma""","""Bmal1""","""0.1uM Dex"""
…,…,…,…
"""J8""","""PhenoV""","""Per2""","""no treatment"""
"""J9""","""PhenoV""","""Per2""","""no treatment"""
"""K15""","""PhenoV""","""Per2""","""0.1uM Dex"""


preprocessing to fix the timepoints

In [8]:
timepoint_col = "Timepoint"
sampling_interval = 3
start_offset = 18
indexing = 1

for f in analysis_ready_dir.glob("2026-03*"):
    if "TIME-CORRECTED" in f.stem:
        continue
    
    curr_lf = (
        pl
        .scan_csv(f, separator="\t")
        .with_columns(
            (start_offset + (pl.col(timepoint_col) - indexing) * sampling_interval)
            .alias(timepoint_col).cast(pl.Float64)
            )
        )
    
    print(f.stem)
    
    out_file = analysis_ready_dir / f"{f.stem}._.TIME-CORRECTED.tsv"
    curr_lf.sink_csv(out_file, separator="\t")

2026-03-10CelPaint_ChromaNoTreatDex
2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED


detrend + denoise again

In [15]:
window_hours = 24
n_timepoints = round(window_hours / sampling_interval)
print(n_timepoints)

well_col = "WellName"

for f in analysis_ready_dir.glob("2026-03*"):
    if 'TIME-CORRECTED' in f.stem:
        out_file = data_dir / f"{f.stem}.parquet"
        print(out_file)
        
        result = run([
            'ccellpaint', 'preprocess',
            str(f), str(out_file),
            '-w', well_col,
            '-t', timepoint_col,
            '--window', str(n_timepoints)
        ], capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"ERROR: {result.stderr}")
        else:
            print("✓ done")

8
/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data/2026-03-10CelPaint_ChromaNoTreatDex._.TIME-CORRECTED.parquet
✓ done
/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/preprocessed_data/2026-03-11CelPaint_PhenovNoTreatDex_HARMONY_FIXED._.TIME-CORRECTED.parquet
✓ done
